In [40]:
import pandas as pd
import re
import glob
import os

def extract_financial_info(text):
    if not isinstance(text, str):
        return None, None, None

    # --- PATTERN 1: Keyword-based (эмитента ... ИНН ... ISIN) ---
    # Example: эмитента ООО ТК "Нафтатранс плюс" ИНН 5404345962 ... ISIN RU000A106Y21
    key_match = re.search(r'эмитента\s+(.*?)\s+ИНН\s+(\d{10,12}).*?ISIN\s+([A-Z0-9]{12})', text)
    if key_match:
        return key_match.group(1).strip(), key_match.group(2), key_match.group(3)

    # --- PATTERN 2: Structured Parentheses (Nested parentheses support) ---
    # Example: (Банк ВТБ (ПАО), 7702070139, RU000A10BJH2, ...)
    parentheses_blocks = re.findall(r'\(([^()]+(?:\([^()]*\)[^()]*)*)\)', text)
    if parentheses_blocks:
        # We look for a block containing a 10-12 digit INN or a 12-char ISIN
        for block in parentheses_blocks:
            parts = [p.strip() for p in block.split(',')]
            if len(parts) >= 3:
                # Validate if it looks like the Russian financial format
                if re.match(r'^\d{10,12}$', parts[1]):
                    name = parts[0].replace('""', '"').strip()
                    return name, parts[1], parts[2]

    # --- PATTERN 3: International/Sovereign (Hyphen based) ---
    # Example: - REPUBLIC OF BELARUS ... (облигация ... / ISIN XS...)
    # Adjusted to catch names even if the date is missing
    int_match = re.search(r'-\s*(.*?)\s*(?:\d{2}/\d{2}/\d{2,4})?\s*\(.*?ISIN\s+([A-Z0-9]{12})\)', text)
    if int_match:
        return int_match.group(1).strip(), None, int_match.group(2)

    # --- PATTERN 4: Disclosure Fallback (No ISIN/INN) ---
    # Example: эмитента ПАО АНК "Башнефть" (выпуск облигаций...)
    dscl_match = re.search(r'эмитента\s+(.*?)\s+\(выпуск облигаций', text)
    if dscl_match:
        return dscl_match.group(1).strip(), None, None

    return None, None, None

# --- FILE PROCESSING ---
file_pattern = os.path.join('CICs', '*_202*.csv')
files = glob.glob(file_pattern)

all_results = []

for file_path in files:
    try:
        df = pd.read_csv(file_path)
        if 'text' not in df.columns:
            continue
            
        # Apply extraction
        extracted_data = df['text'].apply(lambda x: pd.Series(extract_financial_info(x)))
        extracted_data.columns = ['company_name', 'inn', 'isin']
        
        # Merge with original data
        cols_to_keep = ['date', 'link', 'text'] # keeping text for validation
        result_df = pd.concat([df[df.columns.intersection(cols_to_keep)], extracted_data], axis=1)
        all_results.append(result_df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    final_df['date'] = pd.to_datetime(final_df['date'])
    # Output to CSV
    output_name = 'final_extracted_data_2025.csv'
    #final_df.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"Extraction complete. {len(final_df)} rows saved to {output_name}")
else:
    print("No files were processed.")

Extraction complete. 40325 rows saved to final_extracted_data_2025.csv


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2640\2453401915.py:69: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final_df['date'] = pd.to_datetime(final_df['date'])


In [45]:
final_df['month_year'] = final_df['date'].dt.to_period('M')

# Define the month-year sets as Periods
appear = pd.PeriodIndex(['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', 
                         '2025-07', '2025-08', '2025-09', '2025-10', '2025-11',
                         '2026-01', '2026-02', '2025-03', '2025-04', '2025-05',
                         '2025-07', '2025-08', '2025-09'], freq='M')
not_appear = pd.PeriodIndex(['2024-12', '2025-06', '2025-12', '2026-06'], freq='M')

# ISINs present in each appear month
present_in_appear = (final_df[final_df['month_year'].isin(appear)].groupby('isin')['month_year'].apply(set))

# Keep only ISINs that appear in EVERY appear month
isins_in_all_appear = present_in_appear[present_in_appear.apply(lambda s: set(appear).issubset(s))].index

# ISINs that appear in ANY not_appear month (to be excluded)
isins_in_not_appear = set(final_df.loc[final_df['month_year'].isin(not_appear), 'isin'])

# Final filter
target_isins = (set(isins_in_all_appear) - isins_in_not_appear)

result = final_df[final_df['isin'].isin(target_isins)]

In [42]:
result

,date,link,text,company_name,inn,isin,month_year


No ISINs satisfying the requirements